## Imports

In [47]:
import psycopg2 
import pandas as pd
import numpy as np 
from faker import Faker
import os
from dotenv import load_dotenv
from datetime import timedelta

## Database Connection|

In [48]:
load_dotenv()
conn_string = os.getenv('NEON_URL')

conn = psycopg2.connect(conn_string)
cur = conn.cursor()

cur.execute("SELECT version();" )
print(cur.fetchone())

python-dotenv could not parse statement starting at line 7


('PostgreSQL 17.8 (130b160) on aarch64-unknown-linux-gnu, compiled by gcc (Debian 12.2.0-14+deb12u1) 12.2.0, 64-bit',)


## Table Queries

In [49]:
tables_query = """
SELECT 
    table_schema, table_name
FROM 
    information_schema.tables
WHERE 
    table_schema NOT IN ('pg_catalog', 'information_schema')
ORDER BY 
    table_schema, table_name;
"""
df_tables = pd.read_sql(tables_query, conn)
df_tables

/var/folders/5y/20qzsxn55txdbxf0jcdh23h00000gn/T/ipykernel_97716/3562565850.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_tables = pd.read_sql(tables_query, conn)


,table_schema,table_name
0,public,arrival_method
1,public,clinician
2,public,department
3,public,outcome
4,public,patient
5,public,triage_category
6,public,visit


## Variables

In [50]:
# Taken from the nhs admission analysis notebook.

baseline = {
    'avg_ae_attendances': 12006,
    'avg_over_4hrs': 3078,
    'avg_emergency_admissions': 2865,
    'avg_admission_rate': 18.02
}


config = {
    'n_months': 12,
    'start_date': '2025-04-01',
    'department_name': 'Emergency Department',
    'n_patients': 6000,
    'baseline': baseline,
    'n_clinicians': 40,
    'org_name': 'Synthetic NHS Trust',
    'parent_org': 'Synthetic NHS Region',
}

# Hardcoded Lookup Values

In [51]:
department_df = pd.DataFrame({'department_name': [config['department_name']]})
department_df.to_csv('../data/synthetic/_department.csv', index=False)

In [52]:
triage_df = pd.DataFrame([
    {'triage_name': 'Life Threatening Conditions', 'priority': 1},
    {'triage_name': 'Very Urgent', 'priority': 2},
    {'triage_name': 'Urgent', 'priority': 3},
    {'triage_name': 'Not Threatening to Life & Limb', 'priority': 4},
    {'triage_name': 'Not Urgent', 'priority': 5},
])
triage_df.to_csv('../data/synthetic/_triage.csv', index=False)

In [53]:
arrival_method_df = pd.DataFrame([
    {'arrival_method_name': 'Ambulance'},
    {'arrival_method_name': 'Walk-in'},
    {'arrival_method_name': 'GP'},
    {'arrival_method_name': 'NHS 111'},
    {'arrival_method_name': 'Police/Other Emergency Services'},
    {'arrival_method_name': 'Other'},
])

arrival_method_df.to_csv('../data/synthetic/_arrival_method.csv', index=False)

In [54]:
outcome_df = pd.DataFrame([
    {'outcome_name': 'Discharged'},
    {'outcome_name': 'Admitted'},
    {'outcome_name': 'Transferred'},
    {'outcome_name': 'Left Before Treatment Complete'},
])
outcome_df.to_csv('../data/synthetic/_outcome.csv', index=False)

## Synthetic data

In [55]:
fake = Faker('en_GB')
np.random.seed(42)
Faker.seed(42)

### Patient Rows

In [56]:
postcode_areas = ['E', 'N', 'NW', 'SE', 'SW', 'W', 'WC']
sex_values = ['Male', 'Female', 'Other', 'Unknown']
sex_probs = [0.48, 0.48, 0.02, 0.02]

patients = []
for patient_num in range(1, config['n_patients'] + 1):
    age = np.random.randint(1, 100)
    dob = pd.Timestamp('2026-01-01') - pd.DateOffset(years=age) + pd.DateOffset(days=np.random.randint(0, 365))
    sex = np.random.choice(sex_values, p=sex_probs)
    
    patients.append({
        'date_of_birth': dob.date(),
        'sex': sex,
        'postcode_area': np.random.choice(postcode_areas),
    })
    
    
    
patients_df = pd.DataFrame(patients)
patients_df.to_csv('../data/synthetic/_patients.csv', index=False)


### Clinicians

In [58]:
roles_pool = (
    ['Consultant'] * 6 +
    ['Registrar'] * 10 +
    ['Junior Doctor'] * 12 +
    ['Nurse Practitioner'] * 6
)

clinicians = []
for clinician_id, role in enumerate(roles_pool[:config['n_clinicians']], start=1):
    clinicians.append({
        'department_id': 1,
        'clinician_role': role
    })

clinician_df = pd.DataFrame(clinicians)
clinician_df.to_csv('../data/synthetic/_clinicians.csv', index=False)
